# Walk-Forward Retention Next Training Handoff v1

This notebook consolidates the current strict retention gate, SOXLBacktest qlib reference baseline, dashboard replay freshness, and the latest short-training/state-objective blockers into one executable handoff.

Governance:
- Baselines: canonical reference `soxl_backtest_policy_reimplementation_v1`; direct comparison gate `model_retention_walkforward_gate_v1`.
- Role: diagnose and route the next training branch; no model promotion is attempted here.
- Target window: `wf_2020`, `wf_2022`, `wf_2023_2024`, plus current YTD/replay comparison.
- Accepted trade-off: no actor retention unless all required walk-forward windows are present, worst walk-forward drawdown is below 20%, and return is aligned to the current best threshold.
- Changed layer: `selection`, `evaluation`, and `handoff` only.
- Hypothesis: the next useful branch is not longer PPO on the last short-training result; it should first repair metric-aligned official replay checkpoint/BC selection and target scope.


In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for cur in (start, *start.parents):
        if (cur / "AGENTS.md").exists() and (cur / "SPEC.md").exists():
            return cur
    raise FileNotFoundError("Could not find repo root containing AGENTS.md and SPEC.md")


ROOT = find_repo_root(Path.cwd())
CARD = ROOT / "runtime/model_selection/card/walkforward_retention_next_training_handoff_v1"
CARD.mkdir(parents=True, exist_ok=True)


def load_json(rel: str):
    path = ROOT / rel
    if not path.exists():
        return None
    return json.loads(path.read_text())


def summary_of(obj):
    if isinstance(obj, dict) and isinstance(obj.get("summary"), dict):
        return obj["summary"]
    return obj if isinstance(obj, dict) else {}


def write_json(name: str, payload) -> str:
    path = CARD / name
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, ensure_ascii=True) + "\n")
    return str(path.relative_to(ROOT))


def write_csv(name: str, df: pd.DataFrame) -> str:
    path = CARD / name
    df.to_csv(path, index=False)
    return str(path.relative_to(ROOT))

SOURCE_FILES = {
    "model_retention_walkforward_gate": "runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json",
    "strict_candidate_retention_filter": "runtime/model_selection/card/strict_candidate_retention_filter_v1/summary.json",
    "final_walkforward_retention_contract": "runtime/model_selection/card/final_walkforward_retention_contract_v1/summary.json",
    "metric_alignment_gate_flip_audit": "runtime/model_selection/card/metric_alignment_gate_flip_audit_v1/selection.json",
    "sequence_trend_joint_metric_reconciliation": "runtime/model_selection/card/sequence_trend_joint_metric_reconciliation_v1/selection.json",
    "state_objective_checkpoint_selection": "runtime/model_selection/card/state_objective_checkpoint_selection_v1/selection.json",
    "state_head_fold_consensus_target": "runtime/model_selection/card/state_head_fold_consensus_target_v1/selection.json",
    "joint_reward_target_scope_guard": "runtime/model_selection/card/joint_reward_target_scope_guard_v1/summary.json",
    "bull_reentry_bc_warmstart_gate": "runtime/model_selection/card/bull_reentry_bc_warmstart_gate_v1/selection.json",
    "mode_conditioned_drawdown_drift_short_training_screen": "runtime/model_selection/card/mode_conditioned_drawdown_drift_short_training_screen_v1/summary.json",
    "mode_conditioned_drawdown_drift_short_train_summary": "runtime/model/card/mode_conditioned_drawdown_drift_short_train_v1/soxl_mode_conditioned_drawdown_drift_short_train_v1_wf_2020_summary.json",
    "soxlbacktest_qlib_walkforward_baseline": "runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1/summary.json",
    "soxlbacktest_reference_model_score": "runtime/model_selection/card/soxlbacktest_reference_model_score_v1/summary.json",
    "dashboard_model_replay_freshness": "dashboard_data/model_replay_freshness.json",
    "dashboard_v2_kline_difference_refresh": "runtime/model_selection/card/dashboard_v2_kline_difference_refresh_v1/summary.json",
}

sources = {name: load_json(path) for name, path in SOURCE_FILES.items()}
missing_sources = [name for name, payload in sources.items() if payload is None]
source_manifest = {
    name: {
        "path": path,
        "exists": (ROOT / path).exists(),
    }
    for name, path in SOURCE_FILES.items()
}
source_manifest_path = write_json("source_manifest.json", source_manifest)
print(f"repo_root={ROOT}")
print(f"sources={len(source_manifest) - len(missing_sources)} loaded, missing={missing_sources}")
print(f"source_manifest={source_manifest_path}")

repo_root=/Users/rziiiii/dev/workspace
sources=15 loaded, missing=[]
source_manifest=runtime/model_selection/card/walkforward_retention_next_training_handoff_v1/source_manifest.json


## Gate State

The strict gate remains the source of truth for whether any model may be retained as a final actor. The reference baseline can be visible in the dashboard without becoming a retained model.

In [2]:
retention = sources["model_retention_walkforward_gate"] or {}
retention_summary = summary_of(retention)
gate_rows = pd.DataFrame(retention.get("rows", []))
if not gate_rows.empty:
    gate_rows["failures_text"] = gate_rows["failures"].apply(lambda x: ";".join(x) if isinstance(x, list) else ("" if pd.isna(x) else str(x)))
    sort_cols = [c for c in ["gate_pass", "missing_wf_window_count", "wf_drawdown_gap_pct", "best_return_gap_pct"] if c in gate_rows.columns]
    gate_rows_for_display = gate_rows.sort_values(sort_cols, ascending=[False, True, True, True][:len(sort_cols)])
else:
    gate_rows_for_display = gate_rows

strict_summary = summary_of(sources["strict_candidate_retention_filter"])
final_summary = summary_of(sources["final_walkforward_retention_contract"])

thresholds = {
    "required_wf_windows": retention_summary.get("required_wf_windows", []),
    "max_walkforward_drawdown_pct": retention_summary.get("max_walkforward_drawdown_pct"),
    "required_best_return_pct": retention_summary.get("required_best_return_pct"),
    "reference_profile": retention_summary.get("reference_profile"),
    "reference_return_pct": retention_summary.get("reference_return_pct"),
    "current_best_return_pct": retention_summary.get("current_best_return_pct"),
}

gate_snapshot = {
    "retention_summary": retention_summary,
    "strict_summary": strict_summary,
    "final_summary": final_summary,
    "thresholds": thresholds,
}
gate_snapshot_path = write_json("gate_state_snapshot.json", gate_snapshot)
gate_rows_path = write_csv("gate_rows.csv", gate_rows) if not gate_rows.empty else None

print(json.dumps(thresholds, indent=2, sort_keys=True))
print(f"gate_state_snapshot={gate_snapshot_path}")
if gate_rows_path:
    print(f"gate_rows={gate_rows_path}")

display(gate_rows_for_display[[c for c in [
    "profile", "status", "gate_pass", "retention_decision", "ytd_return_pct", "ytd_max_drawdown_pct",
    "worst_wf_max_drawdown_pct", "missing_wf_window_count", "best_return_gap_pct", "wf_drawdown_gap_pct", "failures_text"
] if c in gate_rows_for_display.columns]].head(20))

{
  "current_best_return_pct": 224.361539,
  "max_walkforward_drawdown_pct": 20.0,
  "reference_profile": "soxl_backtest_policy_reimplementation_v1",
  "reference_return_pct": 146.043798,
  "required_best_return_pct": 213.143462,
  "required_wf_windows": [
    "wf_2020",
    "wf_2022",
    "wf_2023_2024"
  ]
}
gate_state_snapshot=runtime/model_selection/card/walkforward_retention_next_training_handoff_v1/gate_state_snapshot.json
gate_rows=runtime/model_selection/card/walkforward_retention_next_training_handoff_v1/gate_rows.csv


,profile,status,gate_pass,retention_decision,ytd_return_pct,ytd_max_drawdown_pct,worst_wf_max_drawdown_pct,missing_wf_window_count,best_return_gap_pct,wf_drawdown_gap_pct,failures_text
0,soxl_backtest_policy_reimplementation_v1,reference,True,keep_reference_baseline,146.043798,13.554089,NaN,0,NaN,NaN,
8,mode_conditioned_rebound_reward_train_v2,ablation,False,retain_as_evidence_only,177.573569,27.278641,21.831720,0,35.569893,1.831720,wf_max_drawdown_ge_20:wf_2020;wf_max_drawdown_...
7,drawdown_rebound_reward_train_v1,ablation,False,retain_as_evidence_only,172.832548,27.310377,23.546830,0,40.310914,3.546830,wf_max_drawdown_ge_20:wf_2020;ytd_return_below...
9,state_conditioned_drawdown_rebound_reward_trai...,ablation,False,retain_as_evidence_only,196.505054,31.390931,27.608986,0,16.638408,7.608986,wf_max_drawdown_ge_20:wf_2020;ytd_return_below...
6,balanced_rule_residual_imitation_v4,candidate,False,retain_as_evidence_only,145.031235,25.999446,35.710864,0,68.112227,15.710864,wf_max_drawdown_ge_20:wf_2020;wf_max_drawdown_...
10,balanced_rule_residual_v19_reg,candidate,False,retain_as_evidence_only,141.992184,25.513909,46.171637,0,71.151278,26.171637,wf_max_drawdown_ge_20:wf_2022;wf_max_drawdown_...
3,balanced_rule_residual_v19,archived,False,retain_as_evidence_only,173.299227,24.399161,46.653957,0,39.844235,26.653957,wf_max_drawdown_ge_20:wf_2020;wf_max_drawdown_...
11,balanced_rule_residual_v19_reg_light,candidate,False,retain_as_evidence_only,164.253638,25.990693,51.596776,0,48.889824,31.596776,wf_max_drawdown_ge_20:wf_2020;wf_max_drawdown_...
5,balanced_rule_residual_v22,canonical,False,retain_as_evidence_only,116.882644,26.278174,54.665610,0,96.260818,34.665610,wf_max_drawdown_ge_20:wf_2020;wf_max_drawdown_...
4,smoke,canonical,False,retain_as_evidence_only,224.361539,30.346650,NaN,3,0.000000,NaN,missing_wf_window:wf_2020;missing_wf_window:wf...


## Dashboard Freshness Contract

Dashboard curves have two different end dates by design: market data and latest signal can use the latest available SOXL date, while qlib daily replay curves stop at the safe backtest boundary when a following calendar boundary is needed.

In [3]:
freshness = sources["dashboard_model_replay_freshness"] or {}
dashboard_summary = sources["dashboard_v2_kline_difference_refresh"] or {}
dashboard_data_status = dashboard_summary.get("data_status", {})

freshness_profiles = pd.DataFrame(freshness.get("profiles", []))
freshness_summary = {
    "latest_data_date": freshness.get("latest_data_date"),
    "safe_backtest_end": freshness.get("safe_backtest_end"),
    "terminal_note": freshness.get("terminal_note"),
    "counts": freshness.get("counts", {}),
    "dashboard_v2_data_status": dashboard_data_status,
}
freshness_path = write_json("dashboard_freshness_snapshot.json", freshness_summary)
print(json.dumps(freshness_summary, indent=2, sort_keys=True))
print(f"dashboard_freshness_snapshot={freshness_path}")

if not freshness_profiles.empty:
    display(freshness_profiles[[c for c in ["profile", "status", "before_end", "after_end", "result", "reason"] if c in freshness_profiles.columns]])

{
  "counts": {
    "current": 12,
    "failed": 0,
    "refreshed": 0,
    "skipped": 0
  },
  "dashboard_v2_data_status": {
    "actor_latest_signal_date": "2026-06-15",
    "actor_replay_end": "2026-06-12",
    "baseline_profile": "soxl_backtest_policy_reimplementation_v1",
    "baseline_replay_end": "2026-06-15",
    "comparison_rows": 112,
    "latest_market_date": "2026-06-15",
    "safe_backtest_end": "2026-06-12"
  },
  "latest_data_date": "2026-06-15",
  "safe_backtest_end": "2026-06-12",
  "terminal_note": "qlib daily backtests require a following calendar boundary, so model replay curves target safe_backtest_end while paper trading/latest signal can use latest_data_date."
}
dashboard_freshness_snapshot=runtime/model_selection/card/walkforward_retention_next_training_handoff_v1/dashboard_freshness_snapshot.json


,profile,status,before_end,after_end,result,reason
0,balanced,archived,2026-06-12,2026-06-12,current,already at safe_backtest_end
1,balanced_macro_v1,candidate,2026-06-12,2026-06-12,current,already at safe_backtest_end
2,balanced_rule_residual_v19,archived,2026-06-12,2026-06-12,current,already at safe_backtest_end
3,smoke,canonical,2026-06-12,2026-06-12,current,already at safe_backtest_end
4,balanced_rule_residual_v22,canonical,2026-06-12,2026-06-12,current,already at safe_backtest_end
5,balanced_rule_residual_imitation_v4,candidate,2026-06-12,2026-06-12,current,already at safe_backtest_end
6,drawdown_rebound_reward_train_v1,ablation,2026-06-12,2026-06-12,current,already at safe_backtest_end
7,mode_conditioned_rebound_reward_train_v2,ablation,2026-06-12,2026-06-12,current,already at safe_backtest_end
8,state_conditioned_drawdown_rebound_reward_trai...,ablation,2026-06-12,2026-06-12,current,already at safe_backtest_end
9,balanced_rule_residual_v19_reg,candidate,2026-06-12,2026-06-12,current,already at safe_backtest_end


## Short-Training Result

The latest drawdown-drift short-training branch must not be extended unless it improves beyond the warm-start/source checkpoint and clears the official replay gate.

In [4]:
short_screen = sources["mode_conditioned_drawdown_drift_short_training_screen"] or {}
short_model = sources["mode_conditioned_drawdown_drift_short_train_summary"] or {}
history = pd.DataFrame(short_model.get("selection_history", []))

if not history.empty:
    best_checkpoint = history.sort_values("score", ascending=False).iloc[0].to_dict()
    post_init = history[history["iter"] > 0]
    best_post_init = post_init.sort_values("score", ascending=False).iloc[0].to_dict() if not post_init.empty else {}
    post_init_score_improved = bool(best_post_init and best_post_init.get("score", float("-inf")) > best_checkpoint.get("score", float("-inf")))
    all_checkpoints_drawdown_fail = bool((history["max_dd"] >= 20.0).all()) if "max_dd" in history else None
else:
    best_checkpoint = {}
    best_post_init = {}
    post_init_score_improved = None
    all_checkpoints_drawdown_fail = None

short_training_assessment = {
    "screen_decision": short_screen.get("decision"),
    "primary_fold": short_screen.get("primary_fold"),
    "primary_gate": short_screen.get("primary_gate"),
    "promotion_allowed": short_screen.get("promotion_allowed"),
    "actor_retention_allowed": short_screen.get("actor_retention_allowed"),
    "training_allowed_flag": short_screen.get("training_allowed"),
    "source_profile": short_screen.get("source_profile"),
    "source_return_pct": short_screen.get("source_return_pct"),
    "short_return_pct": short_screen.get("short_return_pct"),
    "source_max_dd_pct": short_screen.get("source_max_dd_pct"),
    "short_max_dd_pct": short_screen.get("short_max_dd_pct"),
    "return_delta_vs_source_pp": short_screen.get("return_delta_vs_source_pp"),
    "max_dd_improvement_pp": short_screen.get("max_dd_improvement_pp"),
    "best_checkpoint_iter": best_checkpoint.get("iter"),
    "best_checkpoint_score": best_checkpoint.get("score"),
    "best_post_init_iter": best_post_init.get("iter"),
    "best_post_init_score": best_post_init.get("score"),
    "post_init_score_improved": post_init_score_improved,
    "all_checkpoints_drawdown_fail_20pct": all_checkpoints_drawdown_fail,
    "handoff_decision": "stop_longer_ppo_on_this_branch" if best_checkpoint.get("iter") == 0 and not post_init_score_improved else "inspect_before_longer_training",
}
short_training_path = write_json("short_training_noop_assessment.json", short_training_assessment)
history_path = write_csv("short_training_selection_history.csv", history) if not history.empty else None
print(json.dumps(short_training_assessment, indent=2, sort_keys=True))
print(f"short_training_noop_assessment={short_training_path}")
if history_path:
    print(f"short_training_selection_history={history_path}")
    display(history)

{
  "actor_retention_allowed": false,
  "all_checkpoints_drawdown_fail_20pct": true,
  "best_checkpoint_iter": 0.0,
  "best_checkpoint_score": 32.83699360160846,
  "best_post_init_iter": 4.0,
  "best_post_init_score": 24.7608860290418,
  "handoff_decision": "stop_longer_ppo_on_this_branch",
  "max_dd_improvement_pp": 1.7763568394002505e-14,
  "post_init_score_improved": false,
  "primary_fold": "wf_2020",
  "primary_gate": false,
  "promotion_allowed": false,
  "return_delta_vs_source_pp": 0.0,
  "screen_decision": "primary_fold_short_training_failed_no_promotion",
  "short_max_dd_pct": 23.086304019655504,
  "short_return_pct": 50.08332664590836,
  "source_max_dd_pct": 23.086304019655522,
  "source_profile": "mode_conditioned_rebound_reward_train_v2",
  "source_return_pct": 50.08332664590836,
  "training_allowed_flag": true
}
short_training_noop_assessment=runtime/model_selection/card/walkforward_retention_next_training_handoff_v1/short_training_noop_assessment.json
short_training_sele

,iter,score,return,sharpe,max_dd,avg_pos,stress_avg_pos,stress_days,buy_and_hold_return,relative_return
0,0,32.836994,20.567167,1.047852,21.218932,0.307051,0.209682,47.0,98.480336,-77.913169
1,2,4.072797,5.152513,0.383723,21.218932,0.301568,0.223071,47.0,98.480336,-93.327823
2,4,24.760886,16.293177,0.868212,21.218932,0.318925,0.251543,47.0,98.480336,-82.187159


## Blocker Precedence

Older handoff files sometimes say a training branch is allowed next. Those flags are subordinate to hard blockers from the strict gate, metric-alignment mismatch, target-scope guard, and the observed no-op short-training result.

In [5]:
def decision(name: str) -> str | None:
    payload = sources.get(name) or {}
    return summary_of(payload).get("decision")


def next_mechanism(name: str) -> str | None:
    payload = sources.get(name) or {}
    return summary_of(payload).get("next_mechanism")

blockers = [
    {
        "blocker": "strict_retention_gate",
        "active": final_summary.get("final_retained_count", 0) == 0 or strict_summary.get("passing_profile_count", 0) == 0,
        "severity": "hard",
        "evidence": f"final_retained={final_summary.get('final_retained_count')}; strict_passing={strict_summary.get('passing_profile_count')}",
        "action": "do_not_promote_actor_until strict walk-forward gate passes",
    },
    {
        "blocker": "metric_alignment_gate_flip",
        "active": decision("metric_alignment_gate_flip_audit") == "metric_alignment_gate_flip_blocks_training",
        "severity": "hard",
        "evidence": next_mechanism("metric_alignment_gate_flip_audit"),
        "action": "replace source training-selection gates with official replay metrics before new PPO",
    },
    {
        "blocker": "sequence_trend_metric_mismatch",
        "active": decision("sequence_trend_joint_metric_reconciliation") == "sequence_trend_joint_metric_mismatch_no_promotion",
        "severity": "hard",
        "evidence": next_mechanism("sequence_trend_joint_metric_reconciliation"),
        "action": "repair warm-start official evaluation or redefine metric gate first",
    },
    {
        "blocker": "state_head_fold_consensus",
        "active": decision("state_head_fold_consensus_target") == "fold_consensus_removes_all_state_head_lifts_no_training",
        "severity": "hard",
        "evidence": next_mechanism("state_head_fold_consensus_target"),
        "action": "discover stable rebound labels or a new state factor before state-head PPO",
    },
    {
        "blocker": "joint_reward_target_scope",
        "active": decision("joint_reward_target_scope_guard") == "joint_reward_targets_scope_blocked_no_training",
        "severity": "hard",
        "evidence": next_mechanism("joint_reward_target_scope_guard"),
        "action": "regenerate fold-local joint reward targets before weighted BC",
    },
    {
        "blocker": "drawdown_drift_short_training_noop",
        "active": short_training_assessment.get("handoff_decision") == "stop_longer_ppo_on_this_branch",
        "severity": "hard",
        "evidence": "best checkpoint stayed at iter0; post-init checkpoints did not improve score; drawdown remained above 20%",
        "action": "do not extend mode_conditioned_drawdown_drift_short_train_v1 PPO without a changed objective/checkpoint selector",
    },
]
blocker_df = pd.DataFrame(blockers)
blocker_path = write_csv("decision_matrix.csv", blocker_df)
print(f"decision_matrix={blocker_path}")
display(blocker_df)

decision_matrix=runtime/model_selection/card/walkforward_retention_next_training_handoff_v1/decision_matrix.csv


,blocker,active,severity,evidence,action
0,strict_retention_gate,True,hard,final_retained=0; strict_passing=0,do_not_promote_actor_until strict walk-forward...
1,metric_alignment_gate_flip,True,hard,replace source training-selection gates with o...,replace source training-selection gates with o...
2,sequence_trend_metric_mismatch,True,hard,do_not_run_reward_ppo; repair_warm_start_offic...,repair warm-start official evaluation or redef...
3,state_head_fold_consensus,True,hard,discover_stable_rebound_labels_or_new_state_fa...,discover stable rebound labels or a new state ...
4,joint_reward_target_scope,True,hard,regenerate_fold_local_joint_reward_targets,regenerate fold-local joint reward targets bef...
5,drawdown_drift_short_training_noop,True,hard,best checkpoint stayed at iter0; post-init che...,do not extend mode_conditioned_drawdown_drift_...


## Next Action Queue

The queue keeps the current dashboard-visible baseline as a reference while routing model work toward a metric-aligned selection layer before any new long training branch.

In [6]:
baseline_summary = sources["soxlbacktest_qlib_walkforward_baseline"] or {}
reference_score_summary = sources["soxlbacktest_reference_model_score"] or {}
state_objective = sources["state_objective_checkpoint_selection"] or {}
state_objective_summary = summary_of(state_objective)

next_actions = [
    {
        "priority": 1,
        "action_id": "official_replay_metric_aligned_checkpoint_bc_screen_v1",
        "role": "replace_selection",
        "why": "metric-alignment and sequence-trend reconciliation both block new PPO until official replay metrics drive checkpoint/BC selection",
        "input_evidence": "metric_alignment_gate_flip_audit_v1; sequence_trend_joint_metric_reconciliation_v1; soxlbacktest_reference_model_score_v1",
        "expected_artifact": "runtime/model_selection/card/official_replay_metric_aligned_checkpoint_bc_screen_v1/official_replay_metric_aligned_checkpoint_bc_screen_v1.ipynb",
        "gate": "must score candidates on official daily replay return, drawdown, position, and SOXLBacktest buy/sell timing deltas",
        "status": "ready_next",
    },
    {
        "priority": 2,
        "action_id": "regenerate_fold_local_joint_reward_targets_v2",
        "role": "repair_target_scope",
        "why": "joint reward targets are currently scope-blocked, so weighted BC should not proceed on the old targets",
        "input_evidence": "joint_reward_target_scope_guard_v1",
        "expected_artifact": "runtime/model_selection/card/regenerate_fold_local_joint_reward_targets_v2/regenerate_fold_local_joint_reward_targets_v2.ipynb",
        "gate": "no target leakage; fold-local labels only; official replay gate attached before PPO",
        "status": "blocked_until_priority_1_contract",
    },
    {
        "priority": 3,
        "action_id": "stable_rebound_label_or_state_factor_discovery_v1",
        "role": "new_state_feature",
        "why": "state-head fold consensus removed all lifts; lowering thresholds would repeat a failed lever",
        "input_evidence": "state_head_fold_consensus_target_v1",
        "expected_artifact": "runtime/factor/card/stable_rebound_label_or_state_factor_discovery_v1/stable_rebound_label_or_state_factor_discovery_v1.ipynb",
        "gate": "pre-PPO label lift must survive fold consensus",
        "status": "research_next_after_selection_contract",
    },
    {
        "priority": 4,
        "action_id": "stop_mode_conditioned_drawdown_drift_longer_ppo",
        "role": "stop_failed_branch",
        "why": "short training selected iter0 and post-init iterations reduced score without reducing drawdown below 20%",
        "input_evidence": "mode_conditioned_drawdown_drift_short_training_screen_v1; mode_conditioned_drawdown_drift_short_train_v1 summary",
        "expected_artifact": "none; retain evidence only",
        "gate": "requires changed objective or checkpoint selector before another PPO run",
        "status": "closed_no_promotion",
    },
    {
        "priority": 5,
        "action_id": "dashboard_replay_freshness_contract_keep",
        "role": "ui_contract",
        "why": "baseline replay is current to latest market date while actor daily replay stops at safe_backtest_end; UI should expose both dates instead of implying stale data",
        "input_evidence": "dashboard_data/model_replay_freshness.json; dashboard_v2_kline_difference_refresh_v1",
        "expected_artifact": "runtime/model_selection/card/dashboard_v2_kline_difference_refresh_v1/summary.json",
        "gate": "dashboard must show latest_market_date, safe_backtest_end, baseline_replay_end, actor_replay_end, and actor_latest_signal_date",
        "status": "implemented_verify_when_ui_changes_again",
    },
]
next_action_df = pd.DataFrame(next_actions)
next_action_path = write_csv("next_action_queue.csv", next_action_df)
print(f"next_action_queue={next_action_path}")
display(next_action_df)

next_action_queue=runtime/model_selection/card/walkforward_retention_next_training_handoff_v1/next_action_queue.csv


,priority,action_id,role,why,input_evidence,expected_artifact,gate,status
0,1,official_replay_metric_aligned_checkpoint_bc_s...,replace_selection,metric-alignment and sequence-trend reconcilia...,metric_alignment_gate_flip_audit_v1; sequence_...,runtime/model_selection/card/official_replay_m...,must score candidates on official daily replay...,ready_next
1,2,regenerate_fold_local_joint_reward_targets_v2,repair_target_scope,joint reward targets are currently scope-block...,joint_reward_target_scope_guard_v1,runtime/model_selection/card/regenerate_fold_l...,no target leakage; fold-local labels only; off...,blocked_until_priority_1_contract
2,3,stable_rebound_label_or_state_factor_discovery_v1,new_state_feature,state-head fold consensus removed all lifts; l...,state_head_fold_consensus_target_v1,runtime/factor/card/stable_rebound_label_or_st...,pre-PPO label lift must survive fold consensus,research_next_after_selection_contract
3,4,stop_mode_conditioned_drawdown_drift_longer_ppo,stop_failed_branch,short training selected iter0 and post-init it...,mode_conditioned_drawdown_drift_short_training...,none; retain evidence only,requires changed objective or checkpoint selec...,closed_no_promotion
4,5,dashboard_replay_freshness_contract_keep,ui_contract,baseline replay is current to latest market da...,dashboard_data/model_replay_freshness.json; da...,runtime/model_selection/card/dashboard_v2_klin...,"dashboard must show latest_market_date, safe_b...",implemented_verify_when_ui_changes_again


## Summary Artifact

This cell writes the machine-readable summary used by later dashboard/model-selection work.

In [7]:
active_blockers = blocker_df[blocker_df["active"] == True]["blocker"].tolist()
summary = {
    "experiment": "walkforward_retention_next_training_handoff_v1",
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "decision": "no_retained_walkforward_model_route_to_metric_aligned_checkpoint_bc_screen",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "final_retained_count": final_summary.get("final_retained_count", 0),
    "strict_passing_profile_count": strict_summary.get("passing_profile_count", 0),
    "reference_profile": thresholds.get("reference_profile"),
    "reference_return_pct": thresholds.get("reference_return_pct"),
    "required_best_return_pct": thresholds.get("required_best_return_pct"),
    "max_walkforward_drawdown_pct": thresholds.get("max_walkforward_drawdown_pct"),
    "required_wf_windows": thresholds.get("required_wf_windows"),
    "dashboard_freshness": freshness_summary,
    "short_training_assessment": short_training_assessment,
    "active_blockers": active_blockers,
    "next_action": next_actions[0],
    "do_not_run": [
        "do_not_treat_soxlbacktest_qlib_reference_as_promotable_actor",
        "do_not_extend_mode_conditioned_drawdown_drift_short_train_v1_without_changed_objective_or_selector",
        "do_not_lower_state_head_consensus_threshold_to_force_training",
        "do_not_run_sequence_trend_reward_ppo_until official replay metric mismatch is repaired",
    ],
    "sidecars": {
        "source_manifest": source_manifest_path,
        "gate_state_snapshot": gate_snapshot_path,
        "gate_rows": gate_rows_path,
        "dashboard_freshness_snapshot": freshness_path,
        "short_training_noop_assessment": short_training_path,
        "short_training_selection_history": history_path,
        "decision_matrix": blocker_path,
        "next_action_queue": next_action_path,
        "summary": "runtime/model_selection/card/walkforward_retention_next_training_handoff_v1/summary.json",
    },
}
summary_path = write_json("summary.json", summary)
print(json.dumps(summary, indent=2, sort_keys=True))
print(f"summary={summary_path}")

{
  "active_blockers": [
    "strict_retention_gate",
    "metric_alignment_gate_flip",
    "sequence_trend_metric_mismatch",
    "state_head_fold_consensus",
    "joint_reward_target_scope",
    "drawdown_drift_short_training_noop"
  ],
  "actor_retention_allowed": false,
  "created_at": "2026-06-16T02:48:16+00:00",
  "dashboard_freshness": {
    "counts": {
      "current": 12,
      "failed": 0,
      "refreshed": 0,
      "skipped": 0
    },
    "dashboard_v2_data_status": {
      "actor_latest_signal_date": "2026-06-15",
      "actor_replay_end": "2026-06-12",
      "baseline_profile": "soxl_backtest_policy_reimplementation_v1",
      "baseline_replay_end": "2026-06-15",
      "comparison_rows": 112,
      "latest_market_date": "2026-06-15",
      "safe_backtest_end": "2026-06-12"
    },
    "latest_data_date": "2026-06-15",
    "safe_backtest_end": "2026-06-12",
    "terminal_note": "qlib daily backtests require a following calendar boundary, so model replay curves target safe_ba

In [8]:
conclusion_md = f"""
## Conclusion

Decision: `{summary['decision']}`.

No final walk-forward model is retained: `final_retained_count={summary['final_retained_count']}` and `strict_passing_profile_count={summary['strict_passing_profile_count']}`. The SOXLBacktest qlib path remains a buy/sell timing reference baseline, not a promotable actor.

Dashboard freshness is split intentionally: latest market data is `{freshness_summary.get('latest_data_date')}`, safe qlib replay end is `{freshness_summary.get('safe_backtest_end')}`, baseline replay end is `{dashboard_data_status.get('baseline_replay_end')}`, actor replay end is `{dashboard_data_status.get('actor_replay_end')}`, and actor latest signal date is `{dashboard_data_status.get('actor_latest_signal_date')}`.

The latest drawdown-drift short-training branch should not be extended as-is: best checkpoint iter is `{short_training_assessment.get('best_checkpoint_iter')}`, post-init score improved is `{short_training_assessment.get('post_init_score_improved')}`, and all recorded checkpoints still fail the 20% drawdown target: `{short_training_assessment.get('all_checkpoints_drawdown_fail_20pct')}`.

Next action: run `{next_actions[0]['action_id']}` before any new PPO branch, using official daily replay metrics and SOXLBacktest buy/sell timing deltas as the checkpoint/BC selection contract.
""".strip()
(CARD / "conclusion.md").write_text(conclusion_md + "\n")
display(Markdown(conclusion_md))

## Conclusion

Decision: `no_retained_walkforward_model_route_to_metric_aligned_checkpoint_bc_screen`.

No final walk-forward model is retained: `final_retained_count=0` and `strict_passing_profile_count=0`. The SOXLBacktest qlib path remains a buy/sell timing reference baseline, not a promotable actor.

Dashboard freshness is split intentionally: latest market data is `2026-06-15`, safe qlib replay end is `2026-06-12`, baseline replay end is `2026-06-15`, actor replay end is `2026-06-12`, and actor latest signal date is `2026-06-15`.

The latest drawdown-drift short-training branch should not be extended as-is: best checkpoint iter is `0.0`, post-init score improved is `False`, and all recorded checkpoints still fail the 20% drawdown target: `True`.

Next action: run `official_replay_metric_aligned_checkpoint_bc_screen_v1` before any new PPO branch, using official daily replay metrics and SOXLBacktest buy/sell timing deltas as the checkpoint/BC selection contract.